importing Required modules

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn import preprocessing as per
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt 
from sklearn_pandas import DataFrameMapper
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.decomposition import NMF
from lifelines.utils import concordance_index
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,median_absolute_error


import deepsurvk
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNetCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
#READING CSV FILE
df1 = pd.read_csv("data_bcr_clinical_data_patient.csv",na_values='?')
#EXCEPT CLINICAL DATA OTHERS HAVE PATIENT IDs WITH -01, SO ADD -01 AT THE END
df1.at[4,"Patient Identifier"]
def ankfunc(s):
    return s+"-01"
for i in range(4,532):
    df1.at[i,"Patient Identifier"]=ankfunc(df1.at[i,"Patient Identifier"])

#DROP ROWS AND COLUMNS
df1.drop([0,1,2,3] , inplace=True)
df1.set_index("Patient Identifier", inplace=True)

    
df1.replace('unknown',np.nan , inplace=True)
df1.replace('[Not Available]',np.nan , inplace=True)

df1.fillna(df1.mean(), inplace=True)
df1

C:\Users\PRODEE~1\AppData\Local\Temp/ipykernel_21008/2649448865.py:18: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df1.fillna(df1.mean(), inplace=True)


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NaN,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NaN,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [3]:
df1.fillna(method='ffill', inplace=True)
df1.fillna(method='bfill', inplace=True)
df1

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,1:Recurred/Progressed,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NOT HISPANIC OR LATINO,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NOT HISPANIC OR LATINO,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [4]:
df1['Lymph node neck dissection indicator'].replace(['[Not Available]','NO','YES'],['00','1','2'],inplace=True)

df1['Overall Survival Status'].replace(['0:LIVING','1:DECEASED'],['0','1'],inplace=True)
df1['Patient Primary Tumor Site'].replace(['[Not Available]','Buccal Mucosa','Larynx','Oral Cavity','Floor of mouth','Tonsil','Hypopharynx','Alveolar Ridge','Hard Palate','Oropharynx','Lip','Base of tongue','Oral Tongue'],['00','1','2','3','4','5','6','7','8','9','10','11','12'],inplace=True)
df1['Sex'].replace(['[Not Available]','Male','Female'],['00','1','2'],inplace=True)
df1['Race Category'].replace(['[Not Available]','WHITE','BLACK OR AFRICAN AMERICAN','ASIAN','AMERICAN INDIAN OR ALASKA NATIVE'],['00','1','2','3','4'],inplace=True)
df1['Ethnicity Category'].replace(['[Not Available]','NOT HISPANIC OR LATINO','HISPANIC OR LATINO'],['00','1','2'],inplace=True)
df1['Prior Cancer Diagnosis Occurence'].replace(['[Not Available]','No','Yes','Yes, History of Synchronous/Bilateral Malignancy','Yes, History of Prior Malignancy'],['00','1','2','3','4'],inplace=True)
df1['Neoadjuvant Therapy Type Administered Prior To Resection Text'].replace(['[Not Available]','No','Yes'],['00','1','2'],inplace=True)
df1['Vital Status'].replace(['[Not Available]','Dead','Alive'],['00','1','2'],inplace=True)
df1['American Joint Committee on Cancer Publication Version Type'].replace(['[Not Available]','6th','7th','5th','4th'],['00','1','2','3','4'],inplace=True)
df1['American Joint Committee on Cancer Tumor Stage Code'].replace(['[Not Available]','T0','T1','T2','T3','T4','T4a','T4b','TX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Disease Free Status'].replace(['0:DiseaseFree','1:Recurred/Progressed','[Not Available]'],['0','1','00'],inplace=True)
df1['Neoplasm Histologic Grade'].replace(['[Not Available]','G1','G2','G3','GX','G4'],['00','1','2','3','4','5'],inplace=True)
df1['Alcohol History Documented'].replace(['[Not Available]','No','Yes','NO','YES'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','N0','N1','N2','N2a','N2b','N2c','N3','NX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm Disease Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','Discrepancy','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','00','1','2','3','4','5','6'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage'].replace(['[Not Available]','M1','M1','MX','M0'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage'].replace(['[Not Available]','N0','N1','N2a','N2b','N2c','N3','NX','N2'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage'].replace(['[Not Available]','T1','T2','T3','T4a','T4b','TX','T4'],['00','1','2','3','4','5','6','7'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Group Stage'].replace(['[Not Available]','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','1','2','3','4','5','6'],inplace=True)

df1.head()

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,12,1,2,1,1,1,2013,2,2,2,...,66,4,3,4,4,0,0,3.35,0,3.35
TCGA-BA-4074-01,12,1,1,1,1,1,2003,2,1,1,...,69,4,5,3,4,0,1,15.18,1,13.01
TCGA-BA-4075-01,12,1,2,1,2,2,2004,2,1,1,...,49,4,2,4,4,0,1,9.3,1,7.75
TCGA-BA-4076-01,2,1,1,1,1,1,2003,2,1,1,...,39,4,5,3,4,0,1,13.63,1,9.4
TCGA-BA-4077-01,11,2,1,1,2,2,2003,2,1,1,...,45,4,6,5,5,0,1,37.25,1,9.4


In [5]:
#STORING REDUCED DATA TO CSV
cl = pd.DataFrame(df1)
cl.to_csv("CLINICALpreprocessed.csv")
print("Data exported to csv file")

Data exported to csv file


In [6]:
#READING CSV FILE
df =pd.read_csv("data_methylation_hm450.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df2 = df.T
df2.to_csv("Methylation.csv")
df2.head()

Hugo_Symbol,TSEN34,MUSTN1,C3orf16,CKLF,SFRS7,FAM180B,PTPRF,C6orf168,LOC728024,DSTYK,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
TCGA-4P-AA8J-01,0.160658,0.856690,0.689981,0.063268,0.088333,0.738956,0.689032,0.485472,0.847822,0.018144,...,0.021239,0.095778,0.061835,0.043785,0.055341,0.066497,0.080675,0.041548,0.054317,0.098915
TCGA-BA-4074-01,0.172720,0.888797,0.448310,0.095680,0.054274,0.506644,0.842746,0.188788,0.916802,0.018413,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
TCGA-BA-4075-01,0.091838,0.876359,0.336352,0.079018,0.062922,0.475571,0.783786,0.221566,0.792347,0.021204,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
TCGA-BA-4076-01,0.127324,0.911893,0.757925,0.095460,0.073372,0.834641,0.718938,0.791580,0.898727,0.014496,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
TCGA-BA-4077-01,0.132946,0.893790,0.556940,0.074819,0.080927,0.773512,0.392731,0.283078,0.857577,0.018026,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227


In [7]:
# Merge datasets based on the patient identifier
merged_df = pd.merge(cl, df2, left_index=True, right_index=True, how="inner")
df2=merged_df


In [8]:

c2 = pd.DataFrame(df2)
c2.to_csv("Methylationpreprocessed.csv")

In [9]:

# Extract target variable (survival time) from clinical data
y = c2['Overall Survival (Months)']
c2 = c2.drop('Overall Survival (Months)', axis=1)


In [10]:
y.drop(y.index[-1], inplace=True)
dm=c2.iloc[:,:]
#print(d)
dm = dm.reset_index()
M=dm.iloc[1:,1:]
M


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
1,12,1,1,1,1,1,2003,2,1,1,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
2,12,1,2,1,2,2,2004,2,1,1,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
3,2,1,1,1,1,1,2003,2,1,1,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
4,11,2,1,1,2,2,2003,2,1,1,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227
5,2,1,1,1,1,1,2003,2,1,1,...,0.036186,0.065871,0.042553,0.043962,0.036412,0.131708,0.062926,0.043616,0.044998,0.057528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,4,2,1,1,1,1,2009,2,1,1,...,0.019317,0.045418,0.032703,0.032664,0.035815,0.044621,0.080887,0.037113,0.033082,0.048041
524,6,2,1,1,3,1,2011,2,1,2,...,0.030936,0.068638,0.042534,0.043967,0.047700,0.063594,0.094378,0.045045,0.049718,0.084455
525,12,1,1,2,1,1,2013,2,2,2,...,0.024656,0.044608,0.030440,0.047566,0.064576,0.062233,0.051034,0.034986,0.053662,0.051514
526,4,1,1,1,1,1,2012,2,1,2,...,0.026150,0.078658,0.047960,0.048287,0.050811,0.059785,0.077223,0.051496,0.048270,0.059510


In [12]:
#standardize the data
scaler = StandardScaler()
X1 = scaler.fit_transform(M[:])
scaler=per.MinMaxScaler(feature_range=(0,1))
X1=scaler.fit_transform(X1)

#PCA 
# fit pca on data
pca = NMF(n_components=387)
pca.fit(X1)
Z1=pca.transform(X1)


G:\ana\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [13]:
n1 = pd.DataFrame(Z1)
print(n1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
n1["Hugo"]=hugo
n1

(527, 387)


,0,1,2,3,4,5,6,7,8,9,...,378,379,380,381,382,383,384,385,386,Hugo
0,0.006312,0.056827,0.000000,0.010523,0.000000,0.0,0.000000e+00,0.0,0.000056,0.000000,...,0.0,0.000000,0.710344,0.0,0.000000,0.0,0.000000,0.000070,0.0,TCGA-BA-4074-01
1,0.000000,0.000000,0.000000,0.000000,0.034702,0.0,0.000000e+00,0.0,0.000000,0.000000,...,0.0,0.000000,0.826219,0.0,0.000000,0.0,0.000000,0.000000,0.0,TCGA-BA-4075-01
2,0.146835,0.008485,0.000000,0.000000,0.051743,0.0,0.000000e+00,0.0,0.000000,0.000000,...,0.0,0.000000,0.819219,0.0,0.106222,0.0,0.000000,0.344819,0.0,TCGA-BA-4076-01
3,0.098619,0.017379,0.000000,0.000000,0.004548,0.0,2.625820e-02,0.0,0.000347,0.000036,...,0.0,0.000000,1.730977,0.0,0.000000,0.0,0.000000,0.288166,0.0,TCGA-BA-4077-01
4,0.002149,0.073377,0.000000,0.000000,0.000103,0.0,7.124380e-07,0.0,0.000659,0.000000,...,0.0,0.000000,2.615168,0.0,0.401089,0.0,0.000000,0.303643,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.160099,0.000000,0.000000,0.000000,0.000019,0.0,8.683032e-03,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.044728,0.692056,0.0,TCGA-UF-A7JT-01
523,0.161574,0.001133,0.000000,0.022683,0.000049,0.0,6.242272e-03,0.0,0.000000,0.000000,...,0.0,0.002010,0.613914,0.0,0.000000,0.0,0.000000,0.160247,0.0,TCGA-UF-A7JV-01
524,0.005397,0.033679,0.000000,0.000000,0.000000,0.0,5.536089e-02,0.0,0.000000,0.000000,...,0.0,0.000000,2.914087,0.0,0.000000,0.0,0.000000,0.000000,0.0,TCGA-UP-A6WW-01
525,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,0.0,0.007916,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,TCGA-WA-A7GZ-01


In [14]:

lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z1, y)
n_components = Z1.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(20).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zl1 = np.column_stack(columns)
nl1 = pd.DataFrame(Zl1)
print(nl1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nl1["Hugo"]=hugo
nl1

Number of selected features:  20
Selected features:  Int64Index([385, 382, 384, 386, 383, 387, 245, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
            11, 12, 13],
           dtype='int64')
(527, 5)


,0,1,2,3,4,Hugo
0,0.000070,0.000000,0.000000,0.0,0.0,TCGA-BA-4074-01
1,0.000000,0.000000,0.000000,0.0,0.0,TCGA-BA-4075-01
2,0.344819,0.106222,0.000000,0.0,0.0,TCGA-BA-4076-01
3,0.288166,0.000000,0.000000,0.0,0.0,TCGA-BA-4077-01
4,0.303643,0.401089,0.000000,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...
522,0.692056,0.000000,0.044728,0.0,0.0,TCGA-UF-A7JT-01
523,0.160247,0.000000,0.000000,0.0,0.0,TCGA-UF-A7JV-01
524,0.000000,0.000000,0.000000,0.0,0.0,TCGA-UP-A6WW-01
525,0.000000,0.000000,0.000000,0.0,0.0,TCGA-WA-A7GZ-01


In [15]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z1, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z1.shape[1])])
selected_features = coef.abs().nlargest(20).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Ze1 = np.column_stack(columns)
ne1 = pd.DataFrame(Ze1)
ne1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
ne1["Hugo"]=hugo
ne1


Number of selected features:  20
Selected features:  [382, 385, 384, 386, 245, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.000000,0.000070,0.000000,0.0,0.0,0.056827,0.000000,0.010523,0.000000,0.0,...,0.0,0.000056,0.000000,0.000000,0.000000,0.000003,0.000000,0.000150,0.0,TCGA-BA-4074-01
1,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.034702,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.003310,0.000000,0.0,TCGA-BA-4075-01
2,0.106222,0.344819,0.000000,0.0,0.0,0.008485,0.000000,0.000000,0.051743,0.0,...,0.0,0.000000,0.000000,0.000484,0.000000,0.001705,0.004091,0.000000,0.0,TCGA-BA-4076-01
3,0.000000,0.288166,0.000000,0.0,0.0,0.017379,0.000000,0.000000,0.004548,0.0,...,0.0,0.000347,0.000036,0.000191,0.000000,0.000000,0.000000,0.000000,0.0,TCGA-BA-4077-01
4,0.401089,0.303643,0.000000,0.0,0.0,0.073377,0.000000,0.000000,0.000103,0.0,...,0.0,0.000659,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.000000,0.692056,0.044728,0.0,0.0,0.000000,0.000000,0.000000,0.000019,0.0,...,0.0,0.000000,0.000000,0.003210,0.000258,0.002729,0.000000,0.000000,0.0,TCGA-UF-A7JT-01
523,0.000000,0.160247,0.000000,0.0,0.0,0.001133,0.000000,0.022683,0.000049,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.001173,0.007040,0.011687,0.0,TCGA-UF-A7JV-01
524,0.000000,0.000000,0.000000,0.0,0.0,0.033679,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,TCGA-UP-A6WW-01
525,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.007916,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,TCGA-WA-A7GZ-01


In [16]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z1, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z1.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(20)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zrf1 = np.column_stack(columns)
nrf1 = pd.DataFrame(Zrf1)
nrf1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nrf1["Hugo"]=hugo
nrf1

Number of selected features:  20
Selected features:  [202, 17, 42, 365, 283, 78, 173, 224, 3, 28, 63, 104, 39, 193, 376, 231, 34, 176, 361, 43]


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.0,0.000000,0.001832,0.000000e+00,0.0,0.0,0.0,0.0,0.010523,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,TCGA-BA-4074-01
1,0.0,0.000000,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,TCGA-BA-4075-01
2,0.0,0.001293,0.001413,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,TCGA-BA-4076-01
3,0.0,0.000717,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000011,0.0,0.0,0.0,TCGA-BA-4077-01
4,0.0,0.002289,0.000413,9.895545e-07,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.0,0.001033,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000137,0.0,0.000000,0.0,0.0,0.0,TCGA-UF-A7JT-01
523,0.0,0.000000,0.001179,0.000000e+00,0.0,0.0,0.0,0.0,0.022683,0.0,...,0.0,0.0,0.0,0.000012,0.0,0.000000,0.0,0.0,0.0,TCGA-UF-A7JV-01
524,0.0,0.000000,0.000561,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,TCGA-UP-A6WW-01
525,0.0,0.000000,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,TCGA-WA-A7GZ-01


In [17]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=20)
rfe.fit(Z1, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zre1 = np.column_stack(columns)
nre1 = pd.DataFrame(Zre1)
nre1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nre1["Hugo"]=hugo
nre1

Selected Features: [20, 28, 177, 179, 202, 203, 204, 208, 226, 227, 241, 248, 251, 256, 275, 283, 306, 311, 330, 367]


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.000000,0.0,0.000000,1.436860e-07,0.0,0.0,0.000000,0.000180,0.0,0.031934,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-BA-4074-01
1,0.000000,0.0,0.000000,0.000000e+00,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.000003,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-BA-4075-01
2,0.000000,0.0,0.000000,0.000000e+00,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-BA-4076-01
3,0.000000,0.0,0.000000,0.000000e+00,0.0,0.0,0.000008,0.000161,0.0,0.004318,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-BA-4077-01
4,0.000000,0.0,0.000000,0.000000e+00,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000041,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.000000,0.0,0.000000,0.000000e+00,0.0,0.0,0.000000,0.000000,0.0,0.020998,...,0.000038,0.000000,0.000268,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-UF-A7JT-01
523,0.000000,0.0,0.000000,0.000000e+00,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-UF-A7JV-01
524,0.000000,0.0,0.000000,0.000000e+00,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-UP-A6WW-01
525,0.000000,0.0,0.000002,0.000000e+00,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.000022,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,TCGA-WA-A7GZ-01


In [18]:
#READING CSV FILE
df = pd.read_csv("data_RNA_Seq_v2_expression_median.csv")

#DROP ROWS AND COLUMNS
df=df.replace(0,np.nan)
df=df.dropna()
df=df.replace(np.nan,0)
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df4 = df.T
df4.to_csv("RNAb.csv")
df4.head()
merged_df = pd.merge(cl, df4, left_index=True, right_index=True, how="inner")
df4=merged_df

In [19]:
c4 = pd.DataFrame(df4)
c4.to_csv("RNApreprocessed.csv")
y = c4['Overall Survival (Months)']
c4 = c4.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
drn=c4.iloc[:,:]
#print(d)
drn = drn.reset_index()
Rn=drn.iloc[1:,1:]
Rn.head()


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,LOC154274,ZW10,ZWILCH,ZWINT,ZXDB,LOC100130182,ZYG11B,ZYX,FLJ10821,ZZZ3
1,12,1,1,1,1,1,2003,2,1,1,...,311.0030,283.6409,2132.4595,1193.1417,172.2656,380.3717,805.0624,2516.9279,258.5911,1088.3179
2,12,1,2,1,2,2,2004,2,1,1,...,225.1105,512.3945,761.0023,673.1877,172.0488,562.2404,487.7395,5930.0549,292.6437,980.3028
3,2,1,1,1,1,1,2003,2,1,1,...,157.9431,307.4905,480.0682,1032.6643,324.2818,1440.9025,722.5502,2674.5376,672.1763,998.5570
4,11,2,1,1,2,2,2003,2,1,1,...,137.6323,361.4052,1325.3128,1620.3080,210.7796,1423.0029,770.9336,8035.6112,763.2339,692.9740
5,2,1,1,1,1,1,2003,2,1,1,...,241.8520,414.1231,874.1257,1145.1112,372.9953,2634.2473,780.1345,3895.2406,1556.6477,1309.6223


Applying PCA dimensioality reduction technique

In [21]:
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X3 = scaler.fit_transform(Rn[:])
scaler=per.MinMaxScaler(feature_range=(0,1))
X3=scaler.fit_transform(X3)
X3
#fit pca on data
pca = NMF(n_components=323)
pca.fit(X3)
#transform pca
Z3 =pca.transform(X3)
Z3


G:\ana\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.00091607, 0.00388906, ..., 0.        , 0.        ,
        0.07577513],
       ...,
       [0.        , 0.        , 0.01860713, ..., 0.067902  , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.464109  ,
        0.83588005],
       [0.        , 0.00855394, 0.        , ..., 0.17308573, 0.01260298,
        0.        ]])

Applying RFE Feature selection methods

In [22]:
n3 = pd.DataFrame(Z3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
n3["Hugo"]=hugo
n3

,0,1,2,3,4,5,6,7,8,9,...,152,153,154,155,156,157,158,159,160,Hugo
0,0.0,0.000000,0.000000,0.000000,0.008531,0.000000,0.000000,0.000000,0.000000,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4074-01
1,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003773,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4075-01
2,0.0,0.000916,0.003889,0.000000,0.002082,0.000000,0.000000,0.000000,0.000000,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.075775,TCGA-BA-4076-01
3,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.009072,0.000000,0.000000,0.0000,...,0.000000,0.059222,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4077-01
4,0.0,0.024208,0.000000,0.032806,0.012439,0.000000,0.000000,0.030085,0.000000,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.136906,0.056626,0.000000,0.071000,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.0,0.000000,0.000000,0.000000,0.004091,0.000000,0.031087,0.026056,0.000000,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.041041,0.018253,0.000000,TCGA-UF-A7JT-01
515,0.0,0.000580,0.000000,0.000000,0.000000,0.004519,0.027715,0.028615,0.000000,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-UF-A7JV-01
516,0.0,0.000000,0.018607,0.000000,0.000000,0.006666,0.000000,0.000000,0.000000,0.0000,...,0.000000,0.013482,0.000000,0.000000,0.054266,0.000000,0.067902,0.000000,0.000000,TCGA-UP-A6WW-01
517,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.004586,0.028523,0.000000,0.0145,...,0.000000,0.000000,0.000000,0.000000,0.102794,0.000000,0.000000,0.464109,0.835880,TCGA-WA-A7GZ-01


In [23]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z3, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z3.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(20)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zrf3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf3 = pd.DataFrame(Zrf3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nrf3["Hugo"]=hugo
nrf3

Number of selected features:  20
Selected features:  [112, 65, 75, 49, 55, 56, 105, 29, 150, 120, 61, 37, 113, 42, 141, 31, 34, 58, 153, 151]


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000022,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4074-01
1,0.000608,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.004981,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000726,0.000000,0.000000,0.000000,TCGA-BA-4075-01
2,0.000000,0.00000,0.000000,0.002278,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.001335,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4076-01
3,0.000000,0.00000,0.000000,0.000000,0.000000,0.000592,0.000000,0.0,0.027728,0.000000,...,0.000503,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.059222,0.030852,TCGA-BA-4077-01
4,0.000000,0.00000,0.000000,0.000000,0.000138,0.000000,0.000110,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.006983,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.002373,0.00000,0.001515,0.000000,0.000000,0.000000,0.000000,0.0,0.049726,0.000000,...,0.000011,0.000000,0.002617,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-UF-A7JT-01
515,0.003296,0.00088,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.007007,...,0.000216,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-UF-A7JV-01
516,0.000000,0.00000,0.000000,0.000974,0.000000,0.000000,0.003707,0.0,0.000000,0.000000,...,0.000960,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.013482,0.000000,TCGA-UP-A6WW-01
517,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.002137,0.004567,0.000031,0.000000,0.000000,0.000000,0.000000,0.000000,0.115076,TCGA-WA-A7GZ-01


In [24]:
rfe.fit(Z3, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zre3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre3 = pd.DataFrame(Zre3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nre3["Hugo"]=hugo
nre3

Selected Features: [41, 56, 61, 75, 77, 78, 80, 84, 86, 106, 108, 110, 115, 120, 124, 125, 129, 131, 133, 135]


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.000000,0.000000,0.0,0.000000,0.000071,0.000199,0.000000,0.000000,0.000000,0.000026,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4074-01
1,0.000000,0.000000,0.0,0.000000,0.000000,0.000209,0.000000,0.000327,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000036,0.0,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4075-01
2,0.000000,0.000000,0.0,0.000000,0.003269,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000011,0.000000,0.000000,TCGA-BA-4076-01
3,0.000324,0.000592,0.0,0.000000,0.000000,0.000000,0.000053,0.000000,0.000000,0.000027,...,0.004581,0.002750,0.000000,0.000000,0.0,0.000000,0.000000,0.000045,0.000000,TCGA-BA-4077-01
4,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000010,0.000000,0.000224,0.000000,...,0.000000,0.000107,0.000000,0.000000,0.0,0.000000,0.000087,0.000007,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.000261,0.000000,0.0,0.001515,0.000000,0.000000,0.000054,0.000000,0.000371,0.000024,...,0.005490,0.000000,0.000000,0.000002,0.0,0.001014,0.000000,0.000000,0.000579,TCGA-UF-A7JT-01
515,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000097,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.007007,0.000060,0.0,0.000000,0.000056,0.000000,0.009182,TCGA-UF-A7JV-01
516,0.000000,0.000000,0.0,0.000000,0.000000,0.000005,0.000017,0.000000,0.000043,0.000000,...,0.000000,0.001617,0.000000,0.000000,0.0,0.000497,0.000000,0.000000,0.000000,TCGA-UP-A6WW-01
517,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.007158,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.003265,TCGA-WA-A7GZ-01


In [25]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z3, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z3.shape[1])])
selected_features = coef.abs().nlargest(20).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Ze3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne3 = pd.DataFrame(Ze3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
ne3["Hugo"]=hugo
ne3


Number of selected features:  20
Selected features:  [160, 154, 150, 139, 1, 159, 157, 156, 148, 3, 155, 5, 95, 105, 4, 142, 161, 8, 60, 141]


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,Hugo
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008531,0.000000,TCGA-BA-4074-01
1,0.000000,0.000000,0.004981,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000771,0.000000,0.000000,0.000000,TCGA-BA-4075-01
2,0.075775,0.000000,0.000000,0.006749,0.000916,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002082,0.000000,TCGA-BA-4076-01
3,0.000000,0.000000,0.027728,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002757,0.000000,0.000000,0.000877,TCGA-BA-4077-01
4,0.000000,0.000000,0.000000,0.000000,0.024208,0.071000,0.056626,0.136906,0.000000,0.032806,0.000000,0.000000,0.000000,0.000110,0.012439,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.000000,0.000000,0.049726,0.000000,0.000000,0.018253,0.000000,0.000000,0.000261,0.000000,0.000000,0.000000,0.000000,0.000000,0.004091,0.000096,TCGA-UF-A7JT-01
515,0.000000,0.000000,0.000000,0.023277,0.000580,0.000000,0.000000,0.000000,0.024783,0.000000,0.000000,0.004519,0.004726,0.000000,0.000000,0.000000,TCGA-UF-A7JV-01
516,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.054266,0.010982,0.000000,0.000000,0.006666,0.000000,0.003707,0.000000,0.000030,TCGA-UP-A6WW-01
517,0.835880,0.000000,0.000000,0.000000,0.000000,0.464109,0.000000,0.102794,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-WA-A7GZ-01


In [26]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z3, y)
n_components = Z3.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(20).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zl3 = np.column_stack(columns)
nl3 = pd.DataFrame(Zl3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nl3["Hugo"]=hugo
nl3

Number of selected features:  20
Selected features:  Int64Index([160, 154, 139, 159, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,
            15, 16],
           dtype='int64')


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008531,0.000000,0.000000,...,0.000000,0.0000,0.00000,0.0,0.000000,0.000000,0.006319,0.0,0.000000,TCGA-BA-4074-01
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.003773,0.0000,0.00000,0.0,0.000000,0.000000,0.053865,0.0,0.000000,TCGA-BA-4075-01
2,0.075775,0.000000,0.006749,0.000000,0.000916,0.003889,0.000000,0.002082,0.000000,0.000000,...,0.000000,0.0000,0.00000,0.0,0.000000,0.000898,0.002717,0.0,0.000000,TCGA-BA-4076-01
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.009072,...,0.000000,0.0000,0.00000,0.0,0.000000,0.000000,0.006199,0.0,0.000000,TCGA-BA-4077-01
4,0.000000,0.000000,0.000000,0.071000,0.024208,0.000000,0.032806,0.012439,0.000000,0.000000,...,0.000000,0.0000,0.00501,0.0,0.000000,0.000000,0.000000,0.0,0.000934,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.000000,0.000000,0.000000,0.018253,0.000000,0.000000,0.000000,0.004091,0.000000,0.031087,...,0.000000,0.0000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000624,TCGA-UF-A7JT-01
515,0.000000,0.000000,0.023277,0.000000,0.000580,0.000000,0.000000,0.000000,0.004519,0.027715,...,0.000000,0.0000,0.00000,0.0,0.001122,0.000000,0.000000,0.0,0.000000,TCGA-UF-A7JV-01
516,0.000000,0.000000,0.000000,0.000000,0.000000,0.018607,0.000000,0.000000,0.006666,0.000000,...,0.000000,0.0000,0.00000,0.0,0.004191,0.000000,0.000000,0.0,0.000000,TCGA-UP-A6WW-01
517,0.835880,0.000000,0.000000,0.464109,0.000000,0.000000,0.000000,0.000000,0.000000,0.004586,...,0.000000,0.0145,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,TCGA-WA-A7GZ-01


In [27]:
df = pd.read_csv("data_linear_CNA.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df5 = df.T
df5.to_csv("CNA.csv")
df5.head()
merged_df = pd.merge(cl, df5, left_index=True, right_index=True, how="inner")
df5=merged_df
c5 = pd.DataFrame(df5)
c5.to_csv("CNApreprocessed.csv")
y = c5['Overall Survival (Months)']
c5 = c5.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
dcn=c5.iloc[:,:]
#print(d)
dcn = dcn.reset_index()
Cn=dcn.iloc[1:,1:]
Cn.head()
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X4 = scaler.fit_transform(Cn[:])
scaler=per.MinMaxScaler(feature_range=(0,1))
X4=scaler.fit_transform(X4)
X4
#fit pca on data
pca = NMF(n_components=161,random_state=5)
pca.fit(X4)
#transform pca
Z4 =pca.transform(X4)
Z4

n4 = pd.DataFrame(Z4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
n4["Hugo"]=hugo
n4

G:\ana\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


,0,1,2,3,4,5,6,7,8,9,...,152,153,154,155,156,157,158,159,160,Hugo
0,0.001779,0.007962,0.000000,0.000000,0.019182,0.008170,0.000000,0.005082,0.014195,0.006065,...,0.069276,0.000000,0.000000,0.014108,0.003392,0.000000,0.003176,0.005487,0.000000,TCGA-BA-4074-01
1,0.000000,0.000000,0.000000,0.005526,0.000000,0.000000,0.000000,0.018167,0.000000,0.007327,...,0.000000,0.013392,0.110205,0.000000,0.000000,0.030736,0.000000,0.000000,0.000000,TCGA-BA-4075-01
2,0.000000,0.000000,0.000000,0.002561,0.004556,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.004334,0.000000,0.013719,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4076-01
3,0.000000,0.000000,0.011453,0.000000,0.001227,0.004785,0.000000,0.000000,0.000000,0.003889,...,0.033872,0.000000,0.063656,0.000000,0.000000,0.065127,0.000000,0.000000,0.000000,TCGA-BA-4077-01
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.019318,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,0.076001,0.023357,0.014621,0.003045,0.008522,0.006275,0.010029,0.002586,0.005954,0.003685,...,0.041324,0.013232,0.046909,0.014133,0.012546,0.009276,0.032343,0.047255,0.061523,TCGA-UF-A7JT-01
517,0.068300,0.040772,0.011749,0.007676,0.000870,0.004495,0.005177,0.003562,0.003270,0.003737,...,0.023873,0.006281,0.040623,0.033719,0.004919,0.037369,0.049237,0.077564,0.122306,TCGA-UF-A7JV-01
518,0.000000,0.000000,0.000000,0.002956,0.000000,0.009259,0.000000,0.000000,0.000000,0.000000,...,0.016304,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-UP-A6WW-01
519,0.044485,0.000000,0.000000,0.000000,0.001294,0.000000,0.000000,0.013549,0.000000,0.005627,...,0.012574,0.022351,0.008285,0.000000,0.000000,0.012608,0.000000,0.033840,0.000000,TCGA-WA-A7GZ-01


In [33]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z4, y)
n_components = Z4.shape[1]
feature_names = [i for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(20).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zl4 = np.column_stack(columns)
nl4 = pd.DataFrame(Zl4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nl4["Hugo"]=hugo
nl4

Number of selected features:  20
Selected features:  Int64Index([160, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,
            18],
           dtype='int64')


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.000000,0.001779,0.007962,0.000000,0.000000,0.019182,0.008170,0.000000,0.005082,0.014195,...,0.000000,0.004368,0.000692,0.000000,0.002902,0.000000,0.003174,0.000000,0.000000,TCGA-BA-4074-01
1,0.000000,0.000000,0.000000,0.000000,0.005526,0.000000,0.000000,0.000000,0.018167,0.000000,...,0.002397,0.000837,0.000000,0.007043,0.005642,0.000006,0.002249,0.000000,0.000000,TCGA-BA-4075-01
2,0.000000,0.000000,0.000000,0.000000,0.002561,0.004556,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.016316,0.000000,0.001654,0.016372,0.000021,0.000000,0.000000,0.000000,TCGA-BA-4076-01
3,0.000000,0.000000,0.000000,0.011453,0.000000,0.001227,0.004785,0.000000,0.000000,0.000000,...,0.000000,0.005937,0.000000,0.039001,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4077-01
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.019318,0.000000,0.000000,0.000000,...,0.000604,0.024450,0.000000,0.000000,0.000000,0.001850,0.009196,0.000000,0.014838,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,0.061523,0.076001,0.023357,0.014621,0.003045,0.008522,0.006275,0.010029,0.002586,0.005954,...,0.003006,0.000000,0.004264,0.005442,0.005480,0.001191,0.002275,0.000555,0.000019,TCGA-UF-A7JT-01
517,0.122306,0.068300,0.040772,0.011749,0.007676,0.000870,0.004495,0.005177,0.003562,0.003270,...,0.004030,0.000143,0.004762,0.006183,0.006340,0.000888,0.001883,0.000000,0.004288,TCGA-UF-A7JV-01
518,0.000000,0.000000,0.000000,0.000000,0.002956,0.000000,0.009259,0.000000,0.000000,0.000000,...,0.020636,0.002411,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-UP-A6WW-01
519,0.000000,0.044485,0.000000,0.000000,0.000000,0.001294,0.000000,0.000000,0.013549,0.000000,...,0.000000,0.002265,0.000000,0.002579,0.000851,0.004213,0.000000,0.000000,0.003861,TCGA-WA-A7GZ-01


IndexError: index 161 is out of bounds for axis 1 with size 161

In [35]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z4, y)
coef = pd.Series(en.coef_, index=[i for i in range(Z4.shape[1])])
selected_features = coef.abs().nlargest(20).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Ze4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne4 = pd.DataFrame(Ze4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    ne4["Hugo"]=hugo
except Exception:
    pass


Number of selected features:  20
Selected features:  [160, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [36]:
rfe.fit(Z4, y)
selected_features = [i for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zre4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre4 = pd.DataFrame(Zre4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    nre4["Hugo"]=hugo
except Exception:
    pass

Selected Features: [33, 37, 49, 50, 52, 54, 57, 58, 60, 61, 65, 74, 77, 83, 89, 91, 102, 128, 142, 143]


In [37]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z4, y)
importances = rf.feature_importances_
feature_names = [i for i in range(Z4.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(20)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zrf4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf4 = pd.DataFrame(Zrf4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nrf4["Hugo"]=hugo
nrf4

Number of selected features:  20
Selected features:  [134, 160, 48, 26, 150, 89, 74, 143, 34, 102, 40, 77, 0, 91, 141, 125, 3, 80, 127, 139]


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Hugo
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000072,0.000000,0.000100,0.000650,0.000246,...,0.000002,0.001779,0.000078,0.003467,0.000000e+00,0.000000,0.000000,0.001316,0.005630,TCGA-BA-4074-01
1,0.000000,0.000000,0.000477,0.000000,0.010804,0.000215,0.000000,0.000011,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.015104,0.000000e+00,0.005526,0.000060,0.001764,0.004070,TCGA-BA-4075-01
2,0.008277,0.000000,0.000577,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.002561,0.000000,0.000000,0.000000,TCGA-BA-4076-01
3,0.003025,0.000000,0.000833,0.000587,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,4.659623e-03,0.000000,0.000000,0.001974,0.008355,TCGA-BA-4077-01
4,0.000000,0.000000,0.000963,0.008792,0.000000,0.000000,0.000067,0.000000,0.000000,0.000727,...,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.003784,0.003619,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,0.000910,0.061523,0.000185,0.001593,0.000021,0.000133,0.000148,0.000032,0.000905,0.000269,...,0.000002,0.076001,0.000067,0.001329,3.140383e-04,0.003045,0.000418,0.001466,0.005446,TCGA-UF-A7JT-01
517,0.000000,0.122306,0.000000,0.002073,0.012197,0.000164,0.000000,0.000022,0.000679,0.000000,...,0.000021,0.068300,0.000005,0.000000,1.449047e-07,0.007676,0.000000,0.001602,0.005852,TCGA-UF-A7JV-01
518,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.002956,0.000000,0.000000,0.000000,TCGA-UP-A6WW-01
519,0.000000,0.000000,0.000648,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.044485,0.000022,0.003577,0.000000e+00,0.000000,0.000000,0.000000,0.003476,TCGA-WA-A7GZ-01


Merging dataset

In [39]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(n1['Hugo'][i]==n3['Hugo'][j]):
            z.append(n1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==n4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,n1,on='Hugo')
mdf=pd.merge(mdf,n3,on='Hugo')
mdf=pd.merge(mdf,n4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




(512, 26)
(512, 710)
[<keras.callbacks.TerminateOnNaN object at 0x000001BB2EF05370>, <keras.callbacks.ModelCheckpoint object at 0x000001BB2D3BA190>]
Epoch 1/10
1/1 [==============================] - 1s 658ms/step - loss: 150153.0469
Epoch 2/10
1/1 [==============================] - 0s 14ms/step - loss: 156523.1406
Epoch 3/10
1/1 [==============================] - 0s 48ms/step - loss: 139990.4219
Epoch 4/10
1/1 [==============================] - 0s 47ms/step - loss: 121149.6797
Epoch 5/10
1/1 [==============================] - 0s 49ms/step - loss: 116840.9844
Epoch 6/10
1/1 [==============================] - 0s 14ms/step - loss: 119255.1016
Epoch 7/10
1/1 [==============================] - 0s 49ms/step - loss: 103492.2188
Epoch 8/10
1/1 [==============================] - 0s 45ms/step - loss: 100179.4062
Epoch 9/10
1/1 [==============================] - 0s 12ms/step - loss: 115576.8516
Epoch 10/10
1/1 [==============================] - 0s 14ms/step - loss: 110747.3594
DEEPSURV
c-index of

In [64]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

SVR
C-index:  0.9764715875307908
Mean squared error:  5.797651934978568e-05
Mean absolute error:  0.0059180843011056245
Median absolute error:  0.004941792035933722
R-squared:  0.9967626109954071


In [41]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


XGBOOST
C-index:  0.9937170996773645
Mean squared error:  0.0011303112543577538
R-squared:  0.9519825241952404
c_index: 0.9937170996773645
Mean absolute error:  0.006271431550659377
Median absolute error:  0.000990724877386365


In [42]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "base_estimator__max_depth": [1, 2, 3, 4]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9965613856342334
Mean squared error:  0.0009288741709150756
R-squared:  0.9605399018583392
c_index: 0.9965613856342334
Mean absolute error:  0.007196766803078537
Median absolute error:  0.0023054282983580032


In [43]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

C-index:  0.9961793173713703
Mean squared error:  0.00115728571458732
R-squared:  0.9508366048863546
c_index: 0.9961793173713703
Mean absolute error:  0.0068070614495118606
Median absolute error:  0.001144538293631904


In [44]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nl1['Hugo'][i]==nl3['Hugo'][j]):
            z.append(nl1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nl4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nl1,on='Hugo')
mdf=pd.merge(mdf,nl3,on='Hugo')
mdf=pd.merge(mdf,nl4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv)")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




(512, 26)
(512, 46)
[<keras.callbacks.TerminateOnNaN object at 0x000001BB2D3BA370>, <keras.callbacks.ModelCheckpoint object at 0x000001BB2EEBB2E0>]
Epoch 1/10
1/1 [==============================] - 1s 546ms/step - loss: 159458.7344
Epoch 2/10
1/1 [==============================] - 0s 25ms/step - loss: 154180.4844
Epoch 3/10
1/1 [==============================] - 0s 34ms/step - loss: 148468.6875
Epoch 4/10
1/1 [==============================] - 0s 41ms/step - loss: 142622.6250
Epoch 5/10
1/1 [==============================] - 0s 35ms/step - loss: 137777.6719
Epoch 6/10
1/1 [==============================] - 0s 32ms/step - loss: 136315.2500
Epoch 7/10
1/1 [==============================] - 0s 33ms/step - loss: 130579.7188
Epoch 8/10
1/1 [==============================] - 0s 5ms/step - loss: 133141.8438
Epoch 9/10
1/1 [==============================] - 0s 29ms/step - loss: 129045.6953
Epoch 10/10
1/1 [==============================] - 0s 28ms/step - loss: 125389.0469
DeepSurv)
c-index of 

In [45]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

SVR
C-index:  0.9658511722731906
Mean squared error:  0.00011667297244294847
Mean absolute error:  0.0075873330057799025
Median absolute error:  0.005825697557538534
R-squared:  0.9952182595413905


In [46]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


XGBOOST
C-index:  0.9918875297315665
Mean squared error:  0.0003282432179175755
R-squared:  0.9865472367548695
c_index: 0.9918875297315665
Mean absolute error:  0.0046071905091001
Median absolute error:  0.0016245820972411989


In [47]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "base_estimator__max_depth": [1, 2, 3, 4]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9968569486918111
Mean squared error:  0.0010508222982153967
R-squared:  0.9569329606251137
c_index: 0.9968569486918111
Mean absolute error:  0.00617557508513122
Median absolute error:  0.0012408655214956807


In [48]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

C-index:  0.9967720013591573
Mean squared error:  0.0016008723662971488
R-squared:  0.9343896362395666
c_index: 0.9967720013591573
Mean absolute error:  0.007900773896313417
Median absolute error:  0.0006904242194172832


In [49]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(ne1['Hugo'][i]==ne3['Hugo'][j]):
            z.append(ne1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==ne4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,ne1,on='Hugo')
mdf=pd.merge(mdf,ne3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




(512, 26)
(512, 57)
[<keras.callbacks.TerminateOnNaN object at 0x000001BB31497490>, <keras.callbacks.ModelCheckpoint object at 0x000001BB3140C9A0>]
Epoch 1/10
1/1 [==============================] - 1s 537ms/step - loss: 165665.2031
Epoch 2/10
1/1 [==============================] - 0s 28ms/step - loss: 159605.0000
Epoch 3/10
1/1 [==============================] - 0s 23ms/step - loss: 153348.4844
Epoch 4/10
1/1 [==============================] - 0s 32ms/step - loss: 146703.1250
Epoch 5/10
1/1 [==============================] - 0s 32ms/step - loss: 140141.2812
Epoch 6/10
1/1 [==============================] - 0s 26ms/step - loss: 138733.2031
Epoch 7/10
1/1 [==============================] - 0s 38ms/step - loss: 137080.5000
Epoch 8/10
1/1 [==============================] - 0s 42ms/step - loss: 133849.0156
Epoch 9/10
1/1 [==============================] - 0s 0s/step - loss: 135663.6719
Epoch 10/10
1/1 [==============================] - 0s 36ms/step - loss: 133482.7812
DeepSurv
c-index of tr

In [50]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

SVR
C-index:  0.9643433228627218
Mean squared error:  7.320712262246258e-05
Mean absolute error:  0.0065748341787190325
Median absolute error:  0.005015354779650538
R-squared:  0.9961192916002587


In [51]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


XGBOOST
C-index:  0.9905764496137194
Mean squared error:  4.617678587308278e-05
R-squared:  0.9975521693191676
c_index: 0.9905764496137194
Mean absolute error:  0.002998611721482421
Median absolute error:  0.0013895461866889327


In [52]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "base_estimator__max_depth": [1, 2, 3, 4]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9956702606333305
Mean squared error:  0.00020884253428704816
R-squared:  0.9889292605965321
c_index: 0.9956702606333305
Mean absolute error:  0.00728875835930723
Median absolute error:  0.004783635988781718


In [53]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

C-index:  0.997707785041175
Mean squared error:  0.00013884460308079402
R-squared:  0.9926398498106105
c_index: 0.997707785041175
Mean absolute error:  0.003429772514663842
Median absolute error:  0.0006871025908702921


In [54]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nre1['Hugo'][i]==nre3['Hugo'][j]):
            z.append(nre1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nre4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nre1,on='Hugo')
mdf=pd.merge(mdf,nre3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




(512, 26)
(512, 61)
[<keras.callbacks.TerminateOnNaN object at 0x000001BB2EEAC0A0>, <keras.callbacks.ModelCheckpoint object at 0x000001BB3187B880>]
Epoch 1/10
1/1 [==============================] - 1s 514ms/step - loss: 165683.2969
Epoch 2/10
1/1 [==============================] - 0s 23ms/step - loss: 160590.0469
Epoch 3/10
1/1 [==============================] - 0s 36ms/step - loss: 155568.7188
Epoch 4/10
1/1 [==============================] - 0s 32ms/step - loss: 148427.1875
Epoch 5/10
1/1 [==============================] - 0s 33ms/step - loss: 141948.3750
Epoch 6/10
1/1 [==============================] - 0s 30ms/step - loss: 138385.9688
Epoch 7/10
1/1 [==============================] - 0s 33ms/step - loss: 133234.7812
Epoch 8/10
1/1 [==============================] - 0s 34ms/step - loss: 132281.3906
Epoch 9/10
1/1 [==============================] - 0s 36ms/step - loss: 130670.9531
Epoch 10/10
1/1 [==============================] - 0s 41ms/step - loss: 129570.4219
DeepSurv
c-index of 

In [55]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

SVR
C-index:  0.9717245478474994
Mean squared error:  8.00406370811506e-05
Mean absolute error:  0.006307170110437079
Median absolute error:  0.005097179140445322
R-squared:  0.9965406653195005


In [56]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


XGBOOST
C-index:  0.9923579859047296
Mean squared error:  0.0009160014386410243
R-squared:  0.9604106656364365
c_index: 0.9923579859047296
Mean absolute error:  0.005642322473095658
Median absolute error:  0.0013364176569501084


In [57]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "base_estimator__max_depth": [1, 2, 3, 4]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9965186380232657
Mean squared error:  0.0011580618820361793
R-squared:  0.9499488787597908
c_index: 0.9965186380232657
Mean absolute error:  0.006883234971107186
Median absolute error:  0.0019187783716298062


In [58]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

C-index:  0.9971979281650675
Mean squared error:  0.0015190135028717267
R-squared:  0.9343486473589221
c_index: 0.9971979281650675
Mean absolute error:  0.007729032142764691
Median absolute error:  0.0007798709310050458


In [59]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nrf1['Hugo'][i]==nrf3['Hugo'][j]):
            z.append(nrf1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nrf4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nrf1,on='Hugo')
mdf=pd.merge(mdf,nrf3,on='Hugo')
mdf=pd.merge(mdf,nrf4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




(512, 26)
(512, 61)
[<keras.callbacks.TerminateOnNaN object at 0x000001BB2EC89A00>, <keras.callbacks.ModelCheckpoint object at 0x000001BB31C9DDC0>]
Epoch 1/10
1/1 [==============================] - 1s 588ms/step - loss: 162373.0625
Epoch 2/10
1/1 [==============================] - 0s 50ms/step - loss: 158930.6406
Epoch 3/10
1/1 [==============================] - 0s 38ms/step - loss: 153833.2969
Epoch 4/10
1/1 [==============================] - 0s 34ms/step - loss: 147314.0781
Epoch 5/10
1/1 [==============================] - 0s 32ms/step - loss: 140137.9219
Epoch 6/10
1/1 [==============================] - 0s 44ms/step - loss: 132460.2656
Epoch 7/10
1/1 [==============================] - 0s 33ms/step - loss: 131490.2500
Epoch 8/10
1/1 [==============================] - 0s 4ms/step - loss: 132590.8750
Epoch 9/10
1/1 [==============================] - 0s 42ms/step - loss: 126082.1016
Epoch 10/10
1/1 [==============================] - 0s 46ms/step - loss: 123032.9922
DeepSurv
c-index of t

In [60]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

SVR
C-index:  0.9764715875307908
Mean squared error:  5.797651934978568e-05
Mean absolute error:  0.0059180843011056245
Median absolute error:  0.004941792035933722
R-squared:  0.9967626109954071


In [61]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


XGBOOST
C-index:  0.995073473201393
Mean squared error:  2.8975534923406547e-05
R-squared:  0.9983820160434729
c_index: 0.995073473201393
Mean absolute error:  0.0022436971483858134
Median absolute error:  0.0010034144992415311


In [62]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "base_estimator__max_depth": [1, 2, 3, 4]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9945638324980889
Mean squared error:  0.000122261039220581
R-squared:  0.9931729853999199
c_index: 0.9945638324980889
Mean absolute error:  0.005967211654138087
Median absolute error:  0.003844979164330003


In [63]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

C-index:  0.9972819162490444
Mean squared error:  4.562722288536003e-05
R-squared:  0.9974521914848322
c_index: 0.9972819162490444
Mean absolute error:  0.0025962132202048765
Median absolute error:  0.0007613647148144789
